# SparkX-Loop v0.2 — L4 Inference, Telemetry & OpenCode Go Evaluation Suite

This notebook tests your mid-training or completed **60-Effective-Layer Looped Recursive + 3-Tier MoE-Engram** checkpoint on a **Google Colab L4 GPU (24 GB VRAM)** without disturbing your running G4 training session.

- **Default Target Checkpoint Repo**: [`TassP0lar/Spark-X2.6-4B-Loop-mHC-R24-step01850`](https://huggingface.co/TassP0lar/Spark-X2.6-4B-Loop-mHC-R24-step01850)
- **Base Architecture**: `XHToken/Spark-X2.6-4B` (`36 physical layers -> 60 virtual layers` via Sublayer Dual-mHC + 7 Backbone MoE-Engrams = 112 Memory Experts)
- **OpenCode Go Integration**: Uses your **OpenCode Go Coding Plan** (`https://opencode.ai/zen/go/v1`) with `OPENCODE_API_KEY` for automated blind LLM-as-a-Judge grading and side-by-side coding/reasoning verification.

### Quick Start on Colab L4
1. Select **Runtime > Change runtime type > L4 GPU**.
2. Ensure `HF_TOKEN` (or `HF_TOKEM`) and `OPENCODE_API_KEY` (or `OPENCODE_API`) are enabled in **Colab Secrets** (left sidebar key icon).
3. Click **Runtime > Run all**.

In [ ]:
# === Cell 1: Environment Bootstrap & GitHub Repo Sync (L4 GPU) ===
import os, sys, subprocess, shutil
from pathlib import Path

os.environ['HF_HUB_DISABLE_INTERACTIVE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PIP_NO_INPUT'] = '1'

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
REPO_DIR = Path('/content/Spark-X2.6-Recursion')

if Path.cwd().name == 'Spark-X2.6-Recursion' and (Path.cwd() / 'sparkx_loop_core.py').exists():
    REPO_DIR = Path.cwd()
elif Path('/storage/emulated/10/Documents/Spark_X2.6_Loop/sparkx_loop_core.py').exists():
    REPO_DIR = Path('/storage/emulated/10/Documents/Spark_X2.6_Loop')
else:
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=False)
        subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=False)
    os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'accelerate', 'safetensors', 'huggingface_hub', 'openai', 'datasets'
], check=True)

def get_secret(*names, default=''):
    for name in names:
        val = os.environ.get(name, '').strip()
        if val:
            return val
    try:
        from google.colab import userdata
        for name in names:
            try:
                val = userdata.get(name)
                if val and str(val).strip():
                    os.environ[name] = str(val).strip()
                    return str(val).strip()
            except Exception:
                pass
    except Exception:
        pass
    return default

HF_TOKEN = get_secret('HF_TOKEN', 'HF_TOKEM')
OPENCODE_API_KEY = get_secret('OPENCODE_API_KEY', 'OPENCODE_API', 'OPENCODE_GO_API_KEY', 'OPENCODE_KEY')

import torch
assert torch.cuda.is_available(), 'CUDA GPU required! Select L4 GPU in Colab Runtime settings.'
gpu_name = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU Ready: {gpu_name} ({vram_gib:.1f} GiB VRAM)')
print(f'HF_TOKEN loaded: {bool(HF_TOKEN)} | OPENCODE_API_KEY loaded: {bool(OPENCODE_API_KEY)}')

In [ ]:
# === Cell 2: Download & Load TassP0lar/Spark-X2.6-4B-Loop-mHC-R24-step02100 ===
import json
import torch
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from transformers import AutoTokenizer, AutoModelForCausalLM
from sparkx_loop_core import (
    LoopSettings, SparkXLoop, attach_lora, patch_spark_attention_for_long_context
)

# Change this anytime to test newer checkpoints (e.g., step02100 or final R24)
CHECKPOINT_REPO = os.environ.get('CHECKPOINT_REPO', 'TassP0lar/Spark-X2.6-4B-Loop-mHC-R24')
STAGE_SUBFOLDER = os.environ.get('LOOP_STAGE', 'R24')

print(f'Downloading checkpoint from: {CHECKPOINT_REPO} ...')
ckpt_root = Path(snapshot_download(
    repo_id=CHECKPOINT_REPO,
    token=HF_TOKEN or None,
    allow_patterns=['loop_config.json', 'trainable.safetensors', f'stages/{STAGE_SUBFOLDER}/*', '*.json']
))
ckpt_dir = (ckpt_root / 'stages' / STAGE_SUBFOLDER) if (ckpt_root / 'stages' / STAGE_SUBFOLDER / 'loop_config.json').exists() else ckpt_root
meta = json.loads((ckpt_dir / 'loop_config.json').read_text())
print('Loaded loop_config.json:', json.dumps({
    k: meta.get(k) for k in [
        'stage', 'ExportTag', 'train_mode', 'loop_start', 'loop_end', 'loop_passes',
        'engram_buckets', 'engram_dim', 'engram_experts', 'engram_top_k', 'backbone_engram_stride'
    ] if k in meta
}, indent=2))

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
base_id = meta.get('base_id', 'XHToken/Spark-X2.6-4B')
base_rev = meta.get('base_revision', None)

if 'base' not in globals() or 'tok' not in globals():
    print(f'Loading base model & tokenizer ({base_id}) in {dtype} ...')
    tok = AutoTokenizer.from_pretrained(base_id, revision=base_rev, trust_remote_code=True, token=HF_TOKEN or None)
    base = AutoModelForCausalLM.from_pretrained(
        base_id, revision=base_rev, trust_remote_code=True,
        dtype=dtype, device_map={'': 'cuda:0'}, token=HF_TOKEN or None, attn_implementation='eager'
    )
    base.config.use_cache = False
    if meta.get('lora_rank', 0) > 0:
        attach_lora(base, start=4, end=36, rank=meta['lora_rank'], alpha=meta['lora_alpha'])

settings = LoopSettings(
    start=4,
    end=28,
    passes=meta.get('loop_passes', 2),
    eng_buckets=meta.get('engram_buckets', 16384),
    eng_dim=meta.get('engram_dim', 64),
    eng_experts=meta.get('engram_experts', 16),
    eng_top_k=meta.get('engram_top_k', 2),
    eng_expert_rank=meta.get('engram_expert_rank', 16),
    eng_shared_rank=meta.get('engram_shared_rank', 16),
    backbone_engram_stride=meta.get('backbone_engram_stride', 4),
    checkpoint_layers=False,
    dynamic_mhc=meta.get('dynamic_mhc', True),
    sublayer_mhc=meta.get('sublayer_mhc', True),
    pass_adapter_rank=meta.get('pass_adapter_rank', 16),
)

net = SparkXLoop(base, settings, patch_sdpa=True).to('cuda:0')
print('Loading trained weights from trainable.safetensors ...')
state = load_file(str(ckpt_dir / 'trainable.safetensors'), device='cpu')
missing, unexpected = net.load_state_dict(state, strict=False)
del state
torch.cuda.empty_cache()

assert not unexpected, f'Unexpected keys in checkpoint: {unexpected[:10]}'
net.set_loop(
    meta.get('loop_start', 4),
    meta.get('loop_end', 28),
    meta.get('loop_passes', 2),
    use_mhc=meta.get('mhc_enabled', True),
    use_engram=meta.get('engram_enabled', True),
)
net.eval()

alloc_gib = torch.cuda.memory_allocated() / (1024 ** 3)
print(f'Successfully loaded {CHECKPOINT_REPO}!')
print(f'Effective Depth: {net.total_virtual_layers} virtual layers (Physical: {net.n_layers} layers)')
print(f'Frozen keys inherited from base (not in trainable.safetensors): {missing}')
print(f'L4 GPU VRAM Allocated: {alloc_gib:.2f} GiB')

In [ ]:
# === Cell 3: Inspect Learned Sublayer Dual-mHC Gates & 3-Tier MoE-Engram Telemetry ===
import torch

print('=== 1. Sublayer Dual-mHC Recurrent Gate Profile (Layers 4..27) ===')
print(f'{"Layer":>5} | {"Type":<18} | {"attn_gate (tanh)":>16} | {"ffn_gate (tanh)":>15} | {"Pass1 Attn Keep":>15} | {"Pass1 FFN Keep":>14}')
print('-' * 94)

attn_gates, ffn_gates = [], []
with torch.no_grad():
    for idx in range(net.loop_start, net.loop_end):
        ctrl = net.controllers[str(idx)]
        ltype = net.base.model.layers[idx].layer_type
        ag = float(torch.tanh(ctrl.extra_gain.float()).item())
        fg = float(torch.tanh(ctrl.ffn_extra_gain.float()).item())
        attn_gates.append(abs(ag))
        ffn_gates.append(abs(fg))
        # Static + Pass-1 routing bias diagonal keep probability
        m_attn = ctrl.mix_logits.float() + ctrl.pass_routing_bias[1, 0, 4:].reshape(2, 2).float()
        m_ffn = ctrl.ffn_mix_logits.float() + ctrl.pass_routing_bias[1, 1, 4:].reshape(2, 2).float()
        p_attn = float(ctrl.mixing_matrix(base_logits=m_attn)[0, 0].item())
        p_ffn = float(ctrl.mixing_matrix(base_logits=m_ffn)[0, 0].item())
        print(f'{idx:5d} | {ltype:<18} | {ag:+16.4f} | {fg:+15.4f} | {p_attn:15.4f} | {p_ffn:14.4f}')

print('-' * 94)
print(f'Mean |attn_gate| = {sum(attn_gates)/len(attn_gates):.4f} | Mean |ffn_gate| = {sum(ffn_gates)/len(ffn_gates):.4f}\n')

print('=== 2. 3-Tier Memory-Tiered MoE-Engram Modules (Prelude + 6 Macro-Block Anchors) ===')
print(f'{"Location":<18} | {"sig(gain)":>10} | {"||project||_F":>13} | {"||shared_up||_F":>15} | {"Expert Load Min/Max (16 exp)":>28}')
print('-' * 94)
with torch.no_grad():
    all_engrams = [('Prelude (Layer 3)', net.engram)] + [
        (f'Backbone (Layer {k})', mod) for k, mod in net.backbone_engrams.items()
    ]
    for label, mod in all_engrams:
        g = float(torch.sigmoid(mod.gain.float()).item())
        p_norm = float(mod.project.weight.float().norm().item())
        s_norm = float(mod.shared_up.weight.float().norm().item())
        ema = mod.expert_load_ema.float()
        ema_min, ema_max = float(ema.min().item()), float(ema.max().item())
        print(f'{label:<18} | {g:10.4f} | {p_norm:13.4f} | {s_norm:15.4f} | {ema_min:12.4f} .. {ema_max:.4f}')

In [ ]:
# === Cell 5: Formal System Prompts + Agentic Coding Runtime (`SparkXAgenticWorkspace`) + Stateful 60L `_VisitCache` (262K Limit) ===
import html, json, os, re, shlex, shutil, subprocess, sys, tempfile, time
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import TextStreamer
from transformers.cache_utils import DynamicCache
from sparkx_loop_core import (
    patch_spark_attention_for_long_context,
    patch_spark_attention_for_loglinear_kv,
)

DEFAULT_MAX_NEW_TOKENS = 262144  # 262K token limit

# 1. Formal System Prompt for Direct Chat / Code Synthesis
DIRECT_CHAT_SYSTEM_PROMPT = (
    "You are Spark-X2.6-Loop, a 60-effective-layer Looped Recursive + 3-Tier MoE-Engram AI assistant.\n"
    "Follow these core operating principles:\n"
    "1. Inside `<think>...</think>`, formulate a clear, structured architectural plan and verify edge cases.\n"
    "2. Outside `</think>`, deliver the complete, production-ready implementation or answer directly in your response.\n"
    "3. Never use placeholder comments (e.g. `// TODO` or `/* full implementation follows */`) and never invent Windows paths like `C:\\Users\\user\\claude-projects`."
)


# 2. Formal System Prompt Builder for Sandboxed Agentic Workspace Execution
def build_agentic_system_prompt(workspace_root: str, target_file: str = 'index.html') -> str:
    tools_schema = [
        {
            'name': 'list_workspace',
            'description': 'List all files and byte sizes in the active Linux workspace.',
            'parameters': {'type': 'object', 'properties': {'path': {'type': 'string', 'default': '.'}}},
        },
        {
            'name': 'read_file',
            'description': 'Read the contents of a file in the workspace.',
            'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}}, 'required': ['path']},
        },
        {
            'name': 'write_file',
            'description': 'Write complete file contents to disk and automatically run 3D WebGL + JS syntax verification on HTML files.',
            'parameters': {
                'type': 'object',
                'properties': {'path': {'type': 'string'}, 'content': {'type': 'string'}},
                'required': ['path', 'content'],
            },
        },
        {
            'name': 'edit_file',
            'description': 'Replace a specific text span inside an existing workspace file.',
            'parameters': {
                'type': 'object',
                'properties': {'path': {'type': 'string'}, 'old_text': {'type': 'string'}, 'new_text': {'type': 'string'}},
                'required': ['path', 'old_text', 'new_text'],
            },
        },
        {
            'name': 'verify_webgl_page',
            'description': 'Run the 10-point 3D WebGL & JavaScript syntax verifier on an HTML file in the workspace.',
            'parameters': {'type': 'object', 'properties': {'path': {'type': 'string', 'default': target_file}}},
        },
        {
            'name': 'run_shell',
            'description': 'Run a read-only or syntax-check command (ls, wc, head, tail, grep, node --check, python3 -m py_compile) in the workspace.',
            'parameters': {'type': 'object', 'properties': {'command': {'type': 'string'}}, 'required': ['command']},
        },
    ]
    schema_json = json.dumps(tools_schema, indent=2)
    ex_call = '{"name": "...", "arguments": {...}}'
    return (
        f"You are Spark-X2.6-Loop, an autonomous software engineering agent operating inside the Linux workspace `{workspace_root}`.\n"
        f"Primary target deliverable: `{workspace_root}/{target_file}`.\n\n"
        f"### Available Workspace Tools\n"
        f"```json\n{schema_json}\n```\n\n"
        f"### Agent Execution Protocol\n"
        f"1. **Deliberate (`<think>...</think>`)**: Plan the complete software and 3D scene architecture concisely. Do not reference non-existent Windows paths (`C:\\Users\\...`).\n"
        f"2. **Invoke Tools or Emit Deliverable**:\n"
        f"   - To invoke a workspace tool, emit `<tool_call>{ex_call}</tool_call>`.\n"
        f"   - Or emit the complete `{target_file}` directly inside a fenced ```html\n<!DOCTYPE html> ... </html>\n``` code block, which the workspace harness will automatically write to `{workspace_root}/{target_file}` and verify.\n"
        f"3. **No Stubs or Placeholders**: Every file written to `{target_file}` must be 100% self-contained and runnable."
    )


def set_loglinear_kv_runtime(
    enabled: bool = True,
    exact_threshold: int = 2048,
    sink_tokens: int = 32,
    local_window: int = 1024,
    base_block: int = 32,
    hip_top_blocks: int = 8,
):
    '''Toggle runtime-only O(n^2) -> O(n log n) Log-Linear NoPE KV Cache without touching weights.'''
    attn_mod = __import__(type(net.base.model.layers[0].self_attn).__module__, fromlist=['*'])
    attn_mod._sparkx_sdpa_patched = False
    if enabled:
        n = patch_spark_attention_for_loglinear_kv(
            net.base,
            exact_threshold=exact_threshold,
            sink_tokens=sink_tokens,
            local_window=local_window,
            base_block=base_block,
            hip_top_blocks=hip_top_blocks,
        )
        print(f'Runtime O(n log n) Log-Linear NoPE KV Cache ENABLED across {n} layers (threshold={exact_threshold}, sinks={sink_tokens}, local={local_window}, HiP_top={hip_top_blocks}x{base_block}).')
    else:
        n = patch_spark_attention_for_long_context(net.base)
        print(f'Standard 1M-Context Fused SDPA restored across {n} layers.')


def _sample_next_token(
    logits: torch.Tensor,
    out_ids: torch.Tensor,
    temperature: float,
    top_p: float,
    repetition_penalty: float,
) -> torch.Tensor:
    if repetition_penalty != 1.0:
        seen = out_ids[0, -512:].unique()
        score = logits[0, seen]
        logits[0, seen] = torch.where(score < 0, score * repetition_penalty, score / repetition_penalty)
    if temperature <= 1e-5:
        return logits.argmax(dim=-1, keepdim=True)
    probs = F.softmax(logits / temperature, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cum_probs = torch.cumsum(sorted_probs, dim=-1)
    mask = (cum_probs - sorted_probs) > top_p
    sorted_probs[mask] = 0.0
    sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
    sampled = torch.multinomial(sorted_probs, num_samples=1)
    return sorted_idx.gather(-1, sampled)


@torch.no_grad()
def generate_sparkx_loop(
    messages=None,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    enable_thinking: bool = True,
    reasoning_effort: str = 'high',
    temperature: float = 0.0,
    top_p: float = 0.9,
    repetition_penalty: float = 1.04,
    stream: bool = True,
    system_prompt: str | None = None,
    past_key_values=None,
    history_ids: torch.Tensor | None = None,
    append_text: str | None = None,
):
    '''Stateful per-visit KV-cached 60-layer generation supporting system prompts and multi-turn KV cache continuation.'''
    net.eval()
    if past_key_values is None or history_ids is None:
        msg_list = list(messages or [])
        if not any(m.get('role') == 'system' for m in msg_list):
            sys_content = system_prompt if system_prompt is not None else DIRECT_CHAT_SYSTEM_PROMPT
            msg_list = [{'role': 'system', 'content': sys_content}] + msg_list

        prompt_text = tok.apply_chat_template(
            msg_list,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
            reasoning_effort=reasoning_effort,
        )
        if enable_thinking and not prompt_text.rstrip().endswith('<think>'):
            prompt_text = prompt_text + '<think>\n'
        enc = tok(prompt_text, return_tensors='pt', add_special_tokens=False)
        new_chunk_ids = enc['input_ids'].to('cuda:0')
        out = new_chunk_ids
        prev_tok = None
        try:
            cache = DynamicCache(config=net._make_loop_cache_config())
        except TypeError:
            cache = DynamicCache()
        if stream and enable_thinking:
            sys.stdout.write('<think>\n')
            sys.stdout.flush()
    else:
        cache = past_key_values
        if not append_text:
            raise ValueError('append_text is required when continuing from existing past_key_values')
        new_chunk_ids = tok.encode(append_text, add_special_tokens=False, return_tensors='pt').to('cuda:0')
        prev_tok = history_ids[:, -2:] if history_ids.shape[1] >= 2 else history_ids
        out = torch.cat([history_ids, new_chunk_ids], dim=-1)

    prompt_len = out.shape[-1]
    streamer = TextStreamer(tok, skip_prompt=False, skip_special_tokens=False) if stream else None

    t0 = time.perf_counter()
    logits = net.forward_logits(new_chunk_ids, past_key_values=cache, prev_ids=prev_tok)[:, -1, :].float()
    next_id = _sample_next_token(logits, out, temperature, top_p, repetition_penalty)

    if streamer is not None:
        streamer.put(next_id.cpu())
    prev_tok = out[:, -2:]
    out = torch.cat([out, next_id], dim=-1)

    eos_ids = {tok.eos_token_id}
    for extra_eos in ('<|im_end|>', '<|endoftext|>', '<｜end▁of▁sentence｜>'):
        tid = tok.convert_tokens_to_ids(extra_eos)
        if isinstance(tid, int) and tid >= 0:
            eos_ids.add(tid)

    for _ in range(max_new_tokens - 1):
        if int(next_id.item()) in eos_ids:
            break
        logits = net.forward_logits(next_id, past_key_values=cache, prev_ids=prev_tok)[:, -1, :].float()
        next_id = _sample_next_token(logits, out, temperature, top_p, repetition_penalty)

        if streamer is not None:
            streamer.put(next_id.cpu())
        prev_tok = out[:, -2:]
        out = torch.cat([out, next_id], dim=-1)
        if (out.shape[1] - prompt_len) >= 8 and (out.shape[1] % 4 == 0):
            tail_txt = tok.decode(out[0, max(prompt_len, out.shape[1] - 12):], skip_special_tokens=False)
            if '</tool_call>' in tail_txt:
                break

    if streamer is not None:
        streamer.end()

    elapsed = time.perf_counter() - t0
    gen_ids = out[0, prompt_len:]
    raw_text = tok.decode(gen_ids, skip_special_tokens=False)
    if enable_thinking and '</think>' in raw_text and not raw_text.lstrip().startswith('<think>'):
        raw_text = '<think>\n' + raw_text
    think_text, final_answer = '', raw_text
    if '<think>' in raw_text and '</think>' in raw_text:
        think_text = raw_text.split('<think>', 1)[1].split('</think>', 1)[0].strip()
        final_answer = raw_text.split('</think>', 1)[1]
    for eos_str in ('<|im_end|>', '<|endoftext|>', '<｜end▁of▁sentence｜>'):
        final_answer = final_answer.replace(eos_str, '')
    final_answer = final_answer.strip()

    n_gen = int(gen_ids.numel())
    return {
        'raw_text': raw_text,
        'think': think_text,
        'answer': final_answer,
        'tokens': n_gen,
        'tok_per_s': n_gen / max(elapsed, 1e-6),
        'elapsed_s': elapsed,
        'cache': cache,
        'full_ids': out,
    }


class SparkXAgenticWorkspace:
    '''Formal Sandboxed Multi-Turn Agentic Workspace Environment for Spark-X2.6-Loop.

    Provides real filesystem tools (`list_workspace`, `read_file`, `write_file`, `edit_file`),
    sandboxed shell execution (`run_shell`), and automated 3D WebGL + JavaScript syntax verification
    (`verify_webgl_page`), while keeping the 60-layer `_VisitCache` KV state hot across turns!
    '''
    def __init__(self, workspace_dir: str = '/content/sakura_agent_workspace'):
        self.root = Path(workspace_dir).resolve()
        self.root.mkdir(parents=True, exist_ok=True)
        self.tool_log: list[dict] = []

    def _safe_path(self, rel_path: str) -> Path:
        clean = str(rel_path).replace('\\', '/').split('/')[-1] if (':' in str(rel_path) or str(rel_path).startswith('/')) else str(rel_path)
        p = (self.root / clean).resolve()
        if not str(p).startswith(str(self.root)):
            p = self.root / Path(clean).name
        p.parent.mkdir(parents=True, exist_ok=True)
        return p

    def list_workspace(self, path: str = '.') -> dict:
        target = self._safe_path(path) if path != '.' else self.root
        files = []
        for item in sorted(target.rglob('*')):
            if item.is_file():
                files.append({
                    'path': str(item.relative_to(self.root)),
                    'bytes': item.stat().st_size,
                })
        res = {
            'tool': 'list_workspace',
            'workspace_root': str(self.root),
            'files': files,
            'three_js_cdn': 'https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js',
            'status': 'Workspace verified and ready. Write the complete <!DOCTYPE html>...</html> implementation to index.html now.',
        }
        self.tool_log.append(res)
        return res

    def read_file(self, path: str) -> dict:
        p = self._safe_path(path)
        if not p.exists():
            res = {'tool': 'read_file', 'path': p.name, 'error': 'File does not exist yet.'}
        else:
            content = p.read_text(encoding='utf-8', errors='replace')
            res = {'tool': 'read_file', 'path': p.name, 'bytes': len(content), 'preview': content[:1500]}
        self.tool_log.append(res)
        return res

    def write_file(self, path: str, content: str) -> dict:
        p = self._safe_path(path)
        p.write_text(content, encoding='utf-8')
        res = {
            'tool': 'write_file',
            'path': str(p.relative_to(self.root)),
            'absolute_path': str(p),
            'bytes_written': len(content.encode('utf-8')),
            'lines_written': content.count('\n') + 1,
            'status': 'success',
        }
        if p.suffix.lower() in ('.html', '.htm'):
            res['verification'] = self.verify_webgl_page(p.name)
        self.tool_log.append(res)
        return res

    def edit_file(self, path: str, old_text: str, new_text: str) -> dict:
        p = self._safe_path(path)
        if not p.exists():
            return {'tool': 'edit_file', 'path': p.name, 'error': 'File not found.'}
        txt = p.read_text(encoding='utf-8')
        if old_text not in txt:
            return {'tool': 'edit_file', 'path': p.name, 'error': 'old_text not found in file.'}
        updated = txt.replace(old_text, new_text, 1)
        p.write_text(updated, encoding='utf-8')
        res = {'tool': 'edit_file', 'path': p.name, 'bytes_written': len(updated), 'status': 'success'}
        if p.suffix.lower() in ('.html', '.htm'):
            res['verification'] = self.verify_webgl_page(p.name)
        self.tool_log.append(res)
        return res

    def run_shell(self, command: str) -> dict:
        allowed_prefixes = ('ls', 'wc', 'head', 'tail', 'grep', 'cat', 'node', 'python3', 'stat')
        parts = shlex.split(command) if command.strip() else []
        if not parts or parts[0] not in allowed_prefixes:
            return {'tool': 'run_shell', 'command': command, 'error': f'Command must start with one of {allowed_prefixes}'}
        try:
            proc = subprocess.run(
                parts, cwd=str(self.root), capture_output=True, text=True, timeout=10
            )
            res = {
                'tool': 'run_shell',
                'command': command,
                'returncode': proc.returncode,
                'stdout': proc.stdout[:1500],
                'stderr': proc.stderr[:1500],
            }
        except Exception as exc:
            res = {'tool': 'run_shell', 'command': command, 'error': str(exc)}
        self.tool_log.append(res)
        return res

    def verify_webgl_page(self, path: str = 'index.html') -> dict:
        p = self._safe_path(path)
        if not p.exists():
            return {'passed': 0, 'total': 10, 'ok': False, 'error': f'{p.name} does not exist'}
        html_code = p.read_text(encoding='utf-8', errors='replace')
        checks = {
            '1. Valid HTML5 Document (<!DOCTYPE html> ... </html>)': bool(re.search(r'<!DOCTYPE html', html_code, re.I) and re.search(r'</html>', html_code, re.I)),
            '2. Three.js Library Included': ('three' in html_code.lower() and 'THREE' in html_code),
            '3. WebGLRenderer & PerspectiveCamera': ('WebGLRenderer' in html_code and 'PerspectiveCamera' in html_code),
            '4. Voxel BoxGeometry / InstancedMesh': ('BoxGeometry' in html_code or 'InstancedMesh' in html_code),
            '5. Sakura Blossom Palette (Pink/Blossom Hex Colors)': bool(re.search(r'ff[89a-f][0-9a-f]{3}|ffb7c5|ff8da1|ffd1dc|ffc0cb|sakura|blossom', html_code, re.I)),
            '6. Pond / Bridge / Torii / Lantern Voxel Structures': bool(re.search(r'pond|water|bridge|torii|lantern', html_code, re.I)),
            '7. Dynamic Lighting (Ambient/Directional/PointLight)': ('Light' in html_code),
            '8. Falling Petal Animation Loop (requestAnimationFrame)': ('requestAnimationFrame' in html_code),
            '9. Interactive Camera / Mouse / UI Controls': bool(re.search(r'mousedown|pointerdown|wheel|addEventListener|onclick', html_code, re.I)),
            '10. Window Resize Handler': ('resize' in html_code.lower()),
        }
        js_syntax_ok = True
        js_syntax_msg = 'not_checked'
        inline_scripts = [
            m.group(1) for m in re.finditer(r'<script(?![^>]*\bsrc=)[^>]*>([\s\S]*?)</script>', html_code, re.I)
            if m.group(1).strip()
        ]
        if inline_scripts and shutil.which('node'):
            js_file = self.root / '_syntax_check.js'
            js_file.write_text('\n'.join(inline_scripts), encoding='utf-8')
            proc = subprocess.run(['node', '--check', str(js_file)], capture_output=True, text=True)
            js_file.unlink(missing_ok=True)
            js_syntax_ok = (proc.returncode == 0)
            js_syntax_msg = 'OK' if js_syntax_ok else proc.stderr.strip()[:600]

        passed = sum(1 for v in checks.values() if v)
        missing = [k for k, v in checks.items() if not v]
        return {
            'file': p.name,
            'bytes': len(html_code),
            'passed': passed,
            'total': len(checks),
            'js_syntax_ok': js_syntax_ok,
            'js_syntax_msg': js_syntax_msg,
            'missing_checks': missing,
            'checks': checks,
            'ok': (passed >= 8 and checks['1. Valid HTML5 Document (<!DOCTYPE html> ... </html>)'] and js_syntax_ok),
        }

    @staticmethod
    def extract_html(text: str) -> str:
        m = re.search(r'```(?:html|HTML)?\s*(<!DOCTYPE html[\s\S]*?</html>)\s*```', text, re.I)
        if m:
            return m.group(1).strip()
        m2 = re.search(r'(<!DOCTYPE html[\s\S]*?</html>|<html[\s\S]*?</html>)', text, re.I)
        if m2:
            return m2.group(1).strip()
        return ''

    def parse_and_execute_turn(self, assistant_text: str, raw_text: str, target_file: str = 'index.html') -> tuple[list[dict], bool]:
        '''Detect and execute any explicit <tool_call>, XML tool, code block, or implicit workspace-check intent.'''
        results = []

        # 1. Check if complete HTML document was emitted in answer or raw_text
        html_doc = self.extract_html(assistant_text) or self.extract_html(raw_text)
        if html_doc:
            w_res = self.write_file(target_file, html_doc)
            results.append(w_res)
            is_done = bool(w_res.get('verification', {}).get('ok', False))
            return results, is_done

        # 2. Check for explicit <tool_call>{...}</tool_call> JSON blocks
        for tc_match in re.finditer(r'<tool_call>\s*(\{[\s\S]*?\})\s*</tool_call>', assistant_text):
            try:
                payload = json.loads(tc_match.group(1))
                name = str(payload.get('name') or payload.get('tool') or '').lower()
                args = payload.get('arguments') or payload.get('parameters') or {}
                if isinstance(args, str):
                    args = json.loads(args)
                if name in ('list_workspace', 'ls', 'list_dir', 'check_workspace'):
                    results.append(self.list_workspace(args.get('path', '.')))
                elif name in ('read_file', 'read', 'cat'):
                    results.append(self.read_file(args.get('path', target_file)))
                elif name in ('write_file', 'write', 'create_file'):
                    w_res = self.write_file(args.get('path', target_file), args.get('content', ''))
                    results.append(w_res)
                    if w_res.get('verification', {}).get('ok', False):
                        return results, True
                elif name in ('edit_file', 'edit', 'replace'):
                    e_res = self.edit_file(args.get('path', target_file), args.get('old_text', ''), args.get('new_text', ''))
                    results.append(e_res)
                    if e_res.get('verification', {}).get('ok', False):
                        return results, True
                elif name in ('run_shell', 'bash', 'shell'):
                    results.append(self.run_shell(args.get('command', 'ls -la')))
                elif name in ('verify_webgl_page', 'verify'):
                    v_res = self.verify_webgl_page(args.get('path', target_file))
                    results.append(v_res)
                    if v_res.get('ok', False):
                        return results, True
            except Exception as exc:
                results.append({'tool': 'tool_call_parse_error', 'error': str(exc)})

        if results:
            return results, False

        # 3. Implicit Claude-Code Natural Intent Bridge ("Let me check the workspace first.", etc.)
        low = assistant_text.lower()
        if any(k in low for k in ('workspace', 'directory', 'folder', 'check', 'first', 'build this', 'create')):
            ws_res = self.list_workspace('.')
            results.append(ws_res)
            return results, False

        return [self.list_workspace('.')], False

    def run_agentic_task(
        self,
        task_prompt: str,
        target_file: str = 'index.html',
        max_turns: int = 4,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        seed_turn1_result: dict | None = None,
    ) -> dict:
        '''Execute a multi-turn stateful agentic coding session with formal system prompt and live KV-cache reuse across turns.'''
        (self.root / 'README_SPEC.md').write_text(task_prompt, encoding='utf-8')
        agent_sys = build_agentic_system_prompt(str(self.root), target_file=target_file)
        messages = [
            {'role': 'system', 'content': agent_sys},
            {'role': 'user', 'content': task_prompt},
        ]

        cache = None
        full_ids = None
        turns_trace = []
        next_append_text = ''

        for turn_idx in range(1, max_turns + 1):
            print('\n' + '=' * 98)
            print(f'[AGENTIC WORKSPACE TURN {turn_idx}/{max_turns}] Workspace: {self.root}')
            print('=' * 98)

            if turn_idx == 1 and seed_turn1_result is not None and seed_turn1_result.get('raw_text'):
                print('[Reusing existing Turn 1 output already in notebook memory...]')
                res = seed_turn1_result
                cache = res.get('cache')
                full_ids = res.get('full_ids')
            elif turn_idx == 1 or cache is None or full_ids is None:
                res = generate_sparkx_loop(
                    messages,
                    max_new_tokens=max_new_tokens,
                    enable_thinking=True,
                    reasoning_effort='high',
                    temperature=0.0,
                    repetition_penalty=1.03,
                    stream=True,
                    system_prompt=agent_sys,
                )
                cache = res['cache']
                full_ids = res['full_ids']
            else:
                res = generate_sparkx_loop(
                    max_new_tokens=max_new_tokens,
                    enable_thinking=False,
                    temperature=0.0,
                    repetition_penalty=1.03,
                    stream=True,
                    past_key_values=cache,
                    history_ids=full_ids,
                    append_text=next_append_text,
                )
                cache = res['cache']
                full_ids = res['full_ids']

            tool_outputs, is_done = self.parse_and_execute_turn(
                res.get('answer', ''), res.get('raw_text', ''), target_file=target_file
            )
            turns_trace.append({
                'turn': turn_idx,
                'tokens': res.get('tokens', 0),
                'elapsed_s': res.get('elapsed_s', 0.0),
                'tool_outputs': tool_outputs,
                'done': is_done,
            })

            print(f'\n[Agentic Harness — Turn {turn_idx} Tool Execution]:')
            print(json.dumps(tool_outputs, indent=2))

            if is_done:
                print(f'\n[Agentic Task Complete on Turn {turn_idx}!] Verified file: {self.root / target_file}')
                break

            target_path = self.root / target_file
            html_opener = '<!DOCTYPE html>\n<html lang="en">\n'
            if not target_path.exists():
                tool_json = json.dumps(tool_outputs[-1], ensure_ascii=False)
                next_append_text = (
                    f"\n<|user|>\n<tool_response>\n{tool_json}\n</tool_response>\n"
                    f"Workspace `{self.root}` is verified and ready. Now write the complete, self-contained `{target_file}` file.\n"
                    f"<|assistant|>\n<think>\nThe workspace is verified and my 3D Voxel Cherry Blossom Garden architecture is already planned above. "
                    f"I will now output the complete, runnable `{target_file}` file with Three.js r128, InstancedMesh voxel terrain/trees/pond/torii/lanterns, "
                    f"falling sakura petals, orbit camera, and atmosphere toggle.\n</think>\n"
                    f"Here is the complete `{target_file}` implementation for the workspace:\n\n```html\n{html_opener}"
                )
                if cache is None or full_ids is None:
                    messages.append({'role': 'assistant', 'content': res.get('raw_text', '')})
                    messages.append({'role': 'user', 'content': f'<tool_response>\n{tool_json}\n</tool_response>\nNow output the complete `{target_file}` inside ```html ... ```.'})
                    prompt_t2 = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
                    prompt_t2 += f"Here is the complete `{target_file}` implementation:\n\n```html\n{html_opener}"
                    try:
                        cache = DynamicCache(config=net._make_loop_cache_config())
                    except TypeError:
                        cache = DynamicCache()
                    prefix_t2 = prompt_t2[:-len(html_opener)]
                    full_ids = tok.encode(prefix_t2, add_special_tokens=False, return_tensors='pt').to('cuda:0')
                    _ = net.forward_logits(full_ids, past_key_values=cache)
                    next_append_text = html_opener
            else:
                ver = tool_outputs[-1].get('verification', self.verify_webgl_page(target_file))
                next_append_text = (
                    f"\n<|user|>\n<tool_response>\n{json.dumps(ver, ensure_ascii=False)}\n</tool_response>\n"
                    f"Please fix any missing checks or JS syntax issues and output the updated complete `{target_file}` inside ```html\n<!DOCTYPE html>...</html>\n```.\n"
                    f"<|assistant|>\n<think>\nLet's fix the reported verification items and output the complete updated HTML file.\n</think>\n"
                    f"```html\n{html_opener}"
                )

            sys.stdout.write(f'\n[Streaming Agent Turn Continuation...]\n{html_opener}')
            sys.stdout.flush()

        final_file = self.root / target_file
        final_html = final_file.read_text(encoding='utf-8') if final_file.exists() else ''
        if final_html and not final_html.lstrip().lower().startswith('<!doctype html'):
            final_html = '<!DOCTYPE html>\n<html lang="en">\n' + final_html
            final_file.write_text(final_html, encoding='utf-8')

        final_audit = self.verify_webgl_page(target_file) if final_file.exists() else {'passed': 0, 'total': 10, 'ok': False, 'checks': {}}
        return {
            'workspace': str(self.root),
            'target_file': str(final_file),
            'html_code': final_html,
            'verification': final_audit,
            'turns': turns_trace,
        }


set_loglinear_kv_runtime(enabled=True, exact_threshold=2048)
print(f'Formal System Prompts & Agentic Workspace (`SparkXAgenticWorkspace`) Ready! Limit = {DEFAULT_MAX_NEW_TOKENS} tokens (262K).')


In [ ]:
# === Cell 5: Multi-Domain Live-Streaming Benchmark (4096 Token Cap + <think> Reasoning Enabled) ===
TEST_SUITE = [
    {
        'name': '1. PhD / University Physics & Thermodynamics',
        'messages': [
            {'role': 'user', 'content': (
                'A monatomic ideal gas undergoes a reversible polytropic process PV^n = constant '
                'from state (P1, V1, T1) to (P2, V2, T2). Derive the molar heat capacity C_n in terms '
                'of the universal gas constant R and polytropic index n, and find the exact range of n '
                'for which the molar heat capacity C_n is negative.'
            )}
        ],
        'max_new_tokens': 4096,
    },
    {
        'name': '2. Mathematical Reasoning (GSM8K / Competition Math)',
        'messages': [
            {'role': 'user', 'content': (
                'A research lab has two servers, Alpha and Beta. Together they can process a genomic '
                'dataset in 12 hours. If Alpha works alone for 4 hours and then Beta works alone for '
                '9 hours, they finish 50% of the dataset. How many hours would it take Alpha alone '
                'to process the entire dataset? Show step-by-step reasoning.'
            )}
        ],
        'max_new_tokens': 4096,
    },
    {
        'name': '3. Algorithmic Coding (Python)',
        'messages': [
            {'role': 'user', 'content': (
                'Write an efficient Python function `longest_balanced_subarray(nums: list[int]) -> int` '
                'that returns the length of the longest contiguous subarray where the sum of elements '
                'is divisible by k=7, running in O(n) time and O(k) space. Include a brief complexity proof.'
            )}
        ],
        'max_new_tokens': 4096,
    },
    {
        'name': '4. Multi-Turn Conversational Coherence & Deep Deliberation',
        'messages': [
            {'role': 'user', 'content': 'We are designing a recursive transformer where layers [4..28) execute twice using doubly-stochastic 2x2 Sinkhorn gates.'},
            {'role': 'assistant', 'content': '<think>The user is describing a looped recursive transformer with 24 middle layers unrolled for 2 passes and 2-stream manifold-constrained hyper-connections (mHC).</think>That architecture gives you 60 effective layers from 36 physical layers while bounding residual stream norms via the Birkhoff polytope. What aspect are we optimizing next?'},
            {'role': 'user', 'content': 'Why did our mean per-layer |ffn_gate| settle around 0.102 when all 24 layers were looped, compared to 0.246 when only 12 layers were looped?'}
        ],
        'max_new_tokens': 4096,
    },
]

SUITE_RESULTS = []
for item in TEST_SUITE:
    print('=' * 96)
    print(f"TEST: {item['name']} (max_new_tokens={item['max_new_tokens']}, enable_thinking=True, stream=True)")
    print('-' * 96)
    res = generate_sparkx_loop(
        item['messages'],
        max_new_tokens=item['max_new_tokens'],
        enable_thinking=True,
        reasoning_effort='high',
        temperature=0.0,
        stream=True,
    )
    SUITE_RESULTS.append({**item, **res})
    print(f"\n[Completed: {res['num_tokens']} tokens in {res['elapsed_sec']:.2f}s ({res['tok_per_sec']:.1f} tok/s) | <think> chars: {len(res['thinking'])} | answer chars: {len(res['answer'])}]\n")

In [ ]:
# === Cell 6: Live Architecture Ablation (60-Layer Full vs Loop-Only vs 36-Layer Single-Pass) ===
import math
import torch

ablation_messages = [
    {'role': 'user', 'content': 'Prove that if p > 3 is a prime number, then p^2 - 1 is always divisible by 24.'},
    {'role': 'assistant', 'content': (
        '<think>Let p > 3 be prime. Since p is prime and greater than 3, p is odd and not divisible by 3. '
        'Notice p^2 - 1 = (p - 1)(p + 1). Since p is odd, both p - 1 and p + 1 are two consecutive even integers, '
        'so one is divisible by 2 and the other by 4, meaning their product is divisible by 8. '
        'Also, among three consecutive integers p - 1, p, p + 1, one is divisible by 3; since p is prime > 3, '
        'either p - 1 or p + 1 is divisible by 3. Since gcd(3, 8) = 1, (p - 1)(p + 1) is divisible by 24.</think>'
        'Since $p > 3$ is prime, $p^2 - 1 = (p - 1)(p + 1)$ is the product of two consecutive even integers '
        '(giving a factor of $2 \\times 4 = 8$) and one of $p - 1, p + 1$ must be a multiple of $3$ '
        '(since $p$ is not divisible by $3$). Because $\\gcd(8, 3) = 1$, $24 \\mid (p^2 - 1)$.'
    )}
]

full_ids = tok.apply_chat_template(ablation_messages, tokenize=True, return_tensors='pt')
if hasattr(full_ids, 'input_ids'):
    full_ids = full_ids.input_ids
elif isinstance(full_ids, dict):
    full_ids = full_ids['input_ids']
full_ids = full_ids.to('cuda:0')

modes = [
    ('60-Layer Full (R24 Loop + Sublayer Dual-mHC + 7 MoE-Engrams)', 4, 28, 2, True, True),
    ('60-Layer Loop Only (R24 Loop + Dual-mHC, MoE-Engrams Disabled)', 4, 28, 2, True, False),
    ('36-Layer Single-Pass + MoE-Engrams (Recurrent Loop Disabled)', 4, 28, 1, False, True),
    ('36-Layer Pure Backbone (Both Loop & MoE-Engrams Disabled)', 4, 28, 1, False, False),
]

print(f'{"Architecture Mode":<62} | {"Eff. Layers":>11} | {"Loss (NLL)":>10} | {"Perplexity":>10}')
print('-' * 102)
with torch.no_grad():
    for title, s, e, p, use_mhc, use_eng in modes:
        net.set_loop(s, e, p, use_mhc=use_mhc, use_engram=use_eng)
        loss = float(net(full_ids, labels=full_ids)['loss'].item())
        ppl = math.exp(min(loss, 20.0))
        print(f'{title:<62} | {net.total_virtual_layers:11d} | {loss:10.4f} | {ppl:10.4f}')

# Restore full 60-layer R24 + MoE-Engram configuration
net.set_loop(4, 28, 2, use_mhc=True, use_engram=True)

In [ ]:
# === Cell 7: OpenCode Go Coding Plan Judge (https://opencode.ai/zen/go/v1) ===
# Queries /models on your OpenCode Go subscription, handles reasoning_content + content, and grades outputs.
from openai import OpenAI

OPENCODE_API_KEY = get_secret('OPENCODE_API_KEY', 'OPENCODE_API', 'OPENCODE_GO_API_KEY', 'OPENCODE_KEY')
OPENCODE_GO_URLS = [
    'https://opencode.ai/zen/go/v1',
    'https://opencode.ai/zen/v1',
    'https://opencode.ai/inference/openai/v1',
]
PREFERRED_MODELS = [
    'deepseek-v4.1-flash',
    'deepseek-v4-pro',
    'mimo-v2.6-pro',
    'mimo-v2.6-flash',
    'claude-haiku-5.5',
    'qwen3.8-max',
    'qwen3.8-flash',
    'glm-5.3',
    'glm-5.3-flash',
    'kimi-k3.1-pro',
    'kimi-k3',
    'minimax-m3.1-flash',
    'gpt-6-luna',
]

def discover_opencode_go():
    if not OPENCODE_API_KEY:
        print('WARNING: OPENCODE_API_KEY is empty. Check Colab Secrets (key icon on left sidebar).')
        return []
    endpoints = []
    for base_url in OPENCODE_GO_URLS:
        client = OpenAI(api_key=OPENCODE_API_KEY, base_url=base_url, timeout=60.0, max_retries=1)
        try:
            resp_models = client.models.list()
            live_ids = [m.id for m in getattr(resp_models, 'data', []) if getattr(m, 'id', None)]
            if live_ids:
                print(f'Discovered {len(live_ids)} live models at {base_url}: {live_ids[:12]}')
                # Prioritize preferred models that exist in live_ids, then remaining live_ids
                ordered = [m for m in PREFERRED_MODELS if m in live_ids] + [m for m in live_ids if m not in PREFERRED_MODELS]
                endpoints.append((base_url, client, ordered))
                continue
        except Exception as e:
            print(f'Note: /models probe on {base_url}: {type(e).__name__}: {str(e)[:120]}')
        endpoints.append((base_url, client, list(PREFERRED_MODELS)))
    return endpoints

ACTIVE_ENDPOINTS = discover_opencode_go()

def call_opencode_go(prompt: str, max_tokens: int = 2048):
    if not ACTIVE_ENDPOINTS:
        return None, 'OPENCODE_API_KEY secret not set in Colab Secrets.'
    last_err = None
    for base_url, client, candidate_models in ACTIVE_ENDPOINTS:
        for model_id in candidate_models:
            try:
                resp = client.chat.completions.create(
                    model=model_id,
                    messages=[
                        {'role': 'system', 'content': 'You are a concise PhD STEM and software engineering evaluator. Be direct.'},
                        {'role': 'user', 'content': prompt},
                    ],
                    max_tokens=max_tokens,
                    temperature=0.1,
                )
                msg = resp.choices[0].message
                text = (getattr(msg, 'content', None) or '').strip()
                if not text:
                    # Fallback if reasoning model placed output in reasoning_content / reasoning
                    text = (getattr(msg, 'reasoning_content', None) or getattr(msg, 'reasoning', None) or '').strip()
                if text:
                    return f'{model_id} @ {base_url}', text
            except Exception as e:
                last_err = f'{model_id} @ {base_url} -> {type(e).__name__}: {str(e)[:160]}'
                continue
    return None, f'All candidate models failed. Last error: {last_err}'

if OPENCODE_API_KEY and SUITE_RESULTS:
    print('\nRunning OpenCode Go Coding Plan LLM-as-a-Judge across test suite outputs...\n')
    for item in SUITE_RESULTS:
        user_q = item['messages'][-1]['content']
        judge_prompt = (
            f"Evaluate the following model response to the prompt below.\n\n"
            f"[PROMPT]:\n{user_q}\n\n"
            f"[MODEL INTERNAL <think>]:\n{item['thinking'][:4000]}\n\n"
            f"[MODEL FINAL ANSWER]:\n{item['answer'][:4000]}\n\n"
            f"Provide: (1) Score out of 10 for correctness, "
            f"(2) Score out of 10 for reasoning clarity, and (3) a brief 2-sentence critique."
        )
        used_endpoint, verdict = call_opencode_go(judge_prompt)
        print(f"=== Judge Verdict for {item['name']} (via {used_endpoint}) ===")
        print(verdict)
        print()
else:
    print('Skipping OpenCode Go evaluation (add OPENCODE_API_KEY to Colab Secrets to enable).')

In [ ]:
# === Cell 8: Interactive Live-Streaming Prompt Playground (4096+ Token Cap & <think> Enabled) ===
def ask(
    prompt: str,
    max_new_tokens: int = 262144,
    enable_thinking: bool = True,
    reasoning_effort: str = 'high',
    temperature: float = 0.0,
    stream: bool = True,
):
    print(f'[Streaming across {net.total_virtual_layers} virtual layers | max_new_tokens={max_new_tokens} | enable_thinking={enable_thinking}]\n')
    res = generate_sparkx_loop(
        [{'role': 'user', 'content': prompt}],
        max_new_tokens=max_new_tokens,
        enable_thinking=enable_thinking,
        reasoning_effort=reasoning_effort,
        temperature=temperature,
        stream=stream,
    )
    print(f"\n[Speed: {res['tok_per_sec']:.1f} tok/s | Tokens: {res['num_tokens']} | Depth: {net.total_virtual_layers} virtual layers]")
    return res

# Try your own prompt here (streams <think>...</think> + final answer live):
_ = ask('Explain how a 3-tier MoE-Engram memory layer reduces parameter interference during recursive loop finetuning.', max_new_tokens=4096)

In [ ]:
# === Cell 6: Formal Agentic 3D WebGL Build & Verification — Voxel Cherry Blossom (Sakura) Garden ===
import html
from pathlib import Path
from IPython.display import display, HTML

SAKURA_3D_PROMPT = (
    "Build a complete, self-contained, single-file 3D interactive web page (`index.html`, `<!DOCTYPE html> ... </html>`) "
    "that renders a **Voxel Cherry Blossom (Sakura) Garden** using Three.js (`https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js`).\n\n"
    "Your implementation MUST include all of the following visual & interactive features:\n"
    "1. **Procedural Voxel Island & Pond**: A multi-layered voxel garden island (grass, soil, stone border voxels) with a central tranquil reflective water pond and voxel stepping stones.\n"
    "2. **Traditional Red Voxel Torii Gate or Arched Wooden Bridge**: Spanning across or beside the pond.\n"
    "3. **Procedural Voxel Cherry Blossom (Sakura) Trees**: Multiple gnarled dark-wood voxel trunks and branches crowned with lush multi-tone pink (`#ffb7c5`, `#ff8da1`, `#ffd1dc`, `#fff0f5`) voxel blossom canopies.\n"
    "4. **Japanese Stone Lanterns (`Toro`) with Warm Glowing PointLights**: Placed along the garden path casting warm golden light.\n"
    "5. **Animated Falling Sakura Petal Particle System**: Hundreds of pink voxel petals drifting, swaying in a sinusoidal breeze, rotating in 3D, and respawning when they touch the water/ground.\n"
    "6. **Interactive Camera & Lighting HUD**: Custom mouse-drag orbit + scroll-wheel zoom camera controller, a gentle auto-rotate toggle, and a UI button to cycle between **Spring Daylight**, **Golden Sunset**, and **Lantern Night** atmospheres.\n\n"
    "You are running inside a live Agentic Workspace (`/content/sakura_agent_workspace`). "
    "If you check the workspace first, the environment will return the directory state so you can immediately write `index.html`."
)

agent_workspace = SparkXAgenticWorkspace('/content/sakura_agent_workspace')

# If `sakura_res` was already generated in a previous run of this cell (where Turn 1 stopped at "Let me check the workspace first."),
# pass it as `seed_turn1_result` so Turn 2 immediately continues from your 1,585-token <think> plan without waiting 5 minutes!
existing_turn1 = globals().get('sakura_res', None)
if isinstance(existing_turn1, dict) and existing_turn1.get('raw_text') and not agent_workspace.extract_html(existing_turn1.get('raw_text', '')):
    print('Detected previous Turn-1 workspace-check output in `sakura_res` — feeding real <tool_response> to complete Turn 2 immediately!')
else:
    existing_turn1 = None

agent_result = agent_workspace.run_agentic_task(
    SAKURA_3D_PROMPT,
    target_file='index.html',
    max_turns=4,
    max_new_tokens=262144,
    seed_turn1_result=existing_turn1,
)

html_code = agent_result['html_code']
ver = agent_result['verification']

if html_code:
    Path('/content/voxel_cherry_blossom_garden.html').write_text(html_code, encoding='utf-8')
    print(f"\nSaved verified 3D WebGL page to: {agent_result['target_file']} ({len(html_code):,} bytes)")
    print(f"=== Agentic 3D Voxel Sakura Garden Audit ({ver['passed']}/{ver['total']} Passed | JS Syntax: {ver.get('js_syntax_msg', 'OK')}) ===")
    for check_name, ok in ver.get('checks', {}).items():
        print(f"  [{'PASS' if ok else 'WARN'}] {check_name}")

    escaped_srcdoc = html.escape(html_code, quote=True)
    preview_iframe = f'''
    <div style="margin-top:12px; border:2px solid #f48fb1; border-radius:12px; overflow:hidden; background:#111;">
      <div style="padding:8px 14px; background:linear-gradient(90deg,#2d1b2e,#4a2545); color:#ffd1dc; font-family:sans-serif; font-size:14px; display:flex; justify-content:space-between; align-items:center;">
        <span>🌸 <b>Spark-X2.6-4B-Loop-mHC (60-Layer Agentic Workspace)</b> — Live 3D Voxel Cherry Blossom Garden</span>
        <span style="font-size:12px; opacity:0.85;">Drag to Orbit • Scroll to Zoom • Click UI Buttons</span>
      </div>
      <iframe srcdoc="{escaped_srcdoc}" style="width:100%; height:620px; border:none; display:block;" sandbox="allow-scripts allow-same-origin"></iframe>
    </div>
    '''
    display(HTML(preview_iframe))
else:
    print('WARNING: Agentic task did not produce index.html; inspect `agent_result["turns"]`.')
